In [2]:
import pandas as pd
import numpy as np
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec

In [4]:
df = pd.read_csv('data/lemmatized.csv')

In [5]:
tokenized_texts = [word_tokenize(str(t).lower()) for t in df['text']]

In [6]:
w2v_model = Word2Vec(
    sentences=tokenized_texts,
    vector_size=100,      # same scale as TF-IDF
    window=5,
    min_count=2,
    workers=4,
    sg=1
)

w2v_model.save("word2vec.model")

In [7]:
def get_doc_vector(words, model, dim):
    valid_words = [w for w in words if w in model.wv.key_to_index]
    if len(valid_words) == 0:
        return np.zeros(dim)
    return np.mean(model.wv[valid_words], axis=0)

In [8]:
dim = w2v_model.vector_size
vectors = np.vstack([get_doc_vector(words, w2v_model, dim) for words in tokenized_texts])

In [9]:
columns = [f'w2v_{i}' for i in range(dim)]
w2v_df = pd.DataFrame(vectors, columns=columns)
df_w2v = pd.concat([df.reset_index(drop=True), w2v_df.reset_index(drop=True)], axis=1)
df_w2v.to_csv('data/word2vec_dataset.csv', index=False)